<a href="https://colab.research.google.com/github/FattahulAlim/Machine_Learning_Sem5/blob/main/JS06/Praktikum4_244107020018_MFattahulAlim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=b8b2ffdd24ace3ac02d7e53d359b170e7e2fd7ea5fbef35fc729ca2422a5b3a9
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=1a820882bd5fb5fc170ebe7868d5e8f010fd57cdcc7d50ce42d8ce477b59678c
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 26.02905035018921 detik
Query time: 0.00020551681518554688 detik
Neighbors: [911848, 114220, 828681, 249109, 268205] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.014980077743530273 detik
Query time: 0.009620904922485352 detik
Neighbors: [911848 114220 828681 249109 268205] ...

=== HNSW (hnswlib) ===
Build time: 167.47291660308838 detik
Query time: 0.00032329559326171875 detik
Neighbors: [911848 114220 828681 249109 268205] ...


In [ ]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

np.random.seed(42)
n_data = 1_000_000
dim = 5
k = 10

X = np.random.random((n_data, dim)).astype(np.float32)
query = np.random.random((1, dim)).astype(np.float32)

# Versi ternormalisasi (dipakai FAISS untuk cosine)
Xn = X.copy(); faiss.normalize_L2(Xn)
qn = query.copy(); faiss.normalize_L2(qn)

def rata2(fn, n=100):
    t0 = time.perf_counter()
    for _ in range(n):
        r = fn()
    return r, (time.perf_counter() - t0) / n * 1000  # ms

metrics = {
    'euclidean':     dict(annoy='euclidean', hnsw='l2'),
    'cosine':        dict(annoy='angular',   hnsw='cosine'),
    'inner_product': dict(annoy='dot',       hnsw='ip'),
}

hasil = []

for nama, cfg in metrics.items():
    print(f"\n===== METRIC: {nama} =====")

    # ---- FAISS Flat (exact, dijadikan acuan recall) ----
    if nama == 'euclidean':
        fi = faiss.IndexFlatL2(dim); data, q = X, query
    elif nama == 'cosine':
        fi = faiss.IndexFlatIP(dim); data, q = Xn, qn
    else:
        fi = faiss.IndexFlatIP(dim); data, q = X, query

    t0 = time.perf_counter(); fi.add(data); b_faiss = time.perf_counter() - t0
    (D, I), t_faiss = rata2(lambda: fi.search(q, k))
    exact = set(I[0].tolist())
    hasil.append((nama, 'FAISS Flat', b_faiss, t_faiss, 1.0))

    # ---- Annoy ----
    ai = AnnoyIndex(dim, cfg['annoy'])
    t0 = time.perf_counter()
    for i in range(n_data):
        ai.add_item(i, X[i])
    ai.build(10)
    b_annoy = time.perf_counter() - t0
    res, t_annoy = rata2(lambda: ai.get_nns_by_vector(query[0], k))
    hasil.append((nama, 'Annoy', b_annoy, t_annoy, len(exact & set(res)) / k))

    # ---- HNSW ----
    hi = hnswlib.Index(space=cfg['hnsw'], dim=dim)
    t0 = time.perf_counter()
    hi.init_index(max_elements=n_data, ef_construction=200, M=16)
    hi.add_items(X)
    b_hnsw = time.perf_counter() - t0
    hi.set_ef(50)
    (lab, dist), t_hnsw = rata2(lambda: hi.knn_query(query, k=k))
    hasil.append((nama, 'HNSW', b_hnsw, t_hnsw, len(exact & set(lab[0].tolist())) / k))

# ---- Rekap tabel ----
print(f"\n{'Metric':<14}{'Metode':<12}{'Build (s)':>10}{'Query (ms)':>12}{'Recall':>8}")
for m, lib, b, t, r in hasil:
    print(f"{m:<14}{lib:<12}{b:>10.2f}{t:>12.4f}{r:>8.2f}")


===== METRIC: euclidean =====

===== METRIC: cosine =====

===== METRIC: inner_product =====

Metric        Metode       Build (s)  Query (ms)  Recall
euclidean     FAISS Flat        0.03     19.1262    1.00
euclidean     Annoy            24.13      0.0309    1.00
euclidean     HNSW            162.61      0.0157    1.00
cosine        FAISS Flat        0.01      3.8417    1.00
cosine        Annoy            27.75      0.0214    1.00
cosine        HNSW            155.04      0.0155    1.00
inner_product FAISS Flat        0.01      3.5881    1.00
inner_product Annoy            47.25      0.0218    0.00
inner_product HNSW            130.27      0.0203    0.90
